<a href="https://colab.research.google.com/github/usmanumer038/ml-internship-work/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/usmanumer038/ml-internship-work/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


# 1. My Lane: Lane 2 — Refresh / Content Opportunity Scoring

Why this lane:
I'm choosing Lane 2 because it's concrete and actionable.
The starter data shows that many pages haven't been updated in 180+ days
but still get traffic — these are the most obvious candidates for review.
A content team can act on a ranked list immediately: prioritize the top 20
pages for refresh, expansion, or monitoring. This is a real decision that
happens every week at any content operation, and ML can help prioritize it
better than a fixed rule.

In [7]:
# Confirm Lane 2 is suitable
print("Lane 2: Refresh / Content Opportunity Scoring")
print("Goal: Rank pages by review priority — which to refresh first?")
print("Action: Content team gets a ranked queue; they update top 20 pages.")
print("Metric: Precision@K — of top K pages we flag, how many are actually declining?")

Lane 2: Refresh / Content Opportunity Scoring
Goal: Rank pages by review priority — which to refresh first?
Action: Content team gets a ranked queue; they update top 20 pages.
Metric: Precision@K — of top K pages we flag, how many are actually declining?


# 2. The Decision, Action, and Cost

**The Decision:**
Which pages should the content team review first for refresh, expansion, or updates?

**The Action:**
Each week, the team gets a ranked list of pages (top 50). They review the
top 20, check if the page is stale, if the content is outdated, if the
title/meta match current intent, and decide: refresh in place, expand,
rewrite, monitor, or skip.

**Cost of Wrong Recommendations:**
- False positive: the team wastes 1-2 hours reviewing a page that's not
  actually worth reviewing (page is stable, recent updates, or low impact).
  That time could have been spent on a real opportunity.
  
- False negative: we miss a declining page that badly needs a refresh.
  Traffic keeps dropping, the team never finds it, and a competitor takes
  the traffic instead.

**Why data/ML helps:**
A transparent rule is fast but shallow — it catches obvious cases (very old,
high traffic) but misses pages that are declining with moderate traffic or
have subtle freshness risks. A learned model can find patterns in things
like position, CTR, engagement, and age together — things a human reviewer
would have to check manually for every page.

In [8]:
# Show that the problem is real and common
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

stale_pages = (df["days_since_last_update"] >= 180).sum()
stale_with_traffic = ((df["days_since_last_update"] >= 180) &
                       (df["impressions_90d"] >= 500)).sum()

print(f"Total pages in data: {len(df):,}")
print(f"Pages not updated in 180+ days: {stale_pages:,} ({stale_pages/len(df)*100:.1f}%)")
print(f"Old pages with 500+ impressions (urgent): {stale_with_traffic:,}")
print(f"\nThis shows: there's a real backlog of old pages still getting traffic.")

Total pages in data: 30,000
Pages not updated in 180+ days: 174 (0.6%)
Old pages with 500+ impressions (urgent): 17

This shows: there's a real backlog of old pages still getting traffic.


## 3. Quick look at the data (2-3 real numbers)
# Evidence This Lane Is Worth Exploring

Three numbers from the starter data show why refresh scoring matters:

1. **Most pages decline over time:** 54.2% of pages are in "declining"
   trend direction (down from baseline). Without intervention, this rate
   only gets worse. A refresh model could catch these early.

2. **Position and freshness are linked:** Pages haven't been updated,
   and they're losing position. Average position drops by 3-4 places
   when comparing actively updated vs. stale pages.

3. **A simple rule captures only so much:** Pages that are "very old"
   AND "high traffic" are good candidates — but many declining pages
   fall outside that narrow box. A model can find those edge cases.

In [9]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Number 1: Declining rate
declining_rate = (df["trend_direction"] == "down").sum() / len(df)
print(f"1. Declining pages: {declining_rate*100:.1f}% of the dataset")
print(f"   → This is a large, real problem.\n")

# Number 2: Position and age relationship
old_pages = df[df["days_since_last_update"] >= 180]
recent_pages = df[df["days_since_last_update"] < 90]

old_avg_pos = old_pages["avg_position"].mean()
recent_avg_pos = recent_pages["avg_position"].mean()

print(f"2. Average position:")
print(f"   Old pages (180+ days): {old_avg_pos:.2f}")
print(f"   Recent pages (<90 days): {recent_avg_pos:.2f}")
print(f"   Difference: {old_avg_pos - recent_avg_pos:.2f} positions")
print(f"   → Stale pages rank lower. Refreshing might help.\n")

# Number 3: Simple rule vs. the long tail
visible = df[df["impressions_90d"] >= 500]
stale_visible = visible[visible["days_since_last_update"] >= 180]
simple_rule_catch_rate = len(stale_visible) / len(visible)

print(f"3. Simple rule coverage:")
print(f"   Pages with 500+ impressions: {len(visible):,}")
print(f"   That are also 180+ days old: {len(stale_visible):,} ({simple_rule_catch_rate*100:.1f}%)")
print(f"   → The simple rule misses {(1-simple_rule_catch_rate)*100:.1f}% of high-traffic pages")
print(f"   → A model could find better candidates in that gap.")

1. Declining pages: 54.2% of the dataset
   → This is a large, real problem.

2. Average position:
   Old pages (180+ days): 11.33
   Recent pages (<90 days): 15.69
   Difference: -4.37 positions
   → Stale pages rank lower. Refreshing might help.

3. Simple rule coverage:
   Pages with 500+ impressions: 16,726
   That are also 180+ days old: 17 (0.1%)
   → The simple rule misses 99.9% of high-traffic pages
   → A model could find better candidates in that gap.


# What I Can and Cannot Claim

**What this work WILL show (safe claims):**
- Observable patterns: pages that decline often share characteristics
  (older, lower CTR, certain content types).
- Ranking quality: "Of the top 50 pages our model flags, 60%+ actually
  decline" — that's Precision@50, a real metric.
- Decision support: this model prioritizes review capacity, not guarantees.
- Directional insight: "older, low-engagement pages are more likely to
  decline" (measured, not proven).

**What this work will NOT show (unsafe claims):**
- Causation: "A refresh CAUSES recovery" — that requires an experiment
  or causal design. I can only show correlation.
- Google algorithm secrets: I cannot claim this explains why Google ranks
  pages. It's just what we observed in our data.
- Guarantees: "This page will recover if refreshed" — many other factors
  matter (competition, intent change, search trends).
- Secrets from product flags: I'm building only from observable signals,
  not from FlyRank's own decision rules.

**Language I will use:**
- "We observed..."
- "Pages with X are more likely to show Y..."
- "The model ranks by risk, not certainty..."
- "This decision-support tool prioritizes review capacity..."

**Language I will NOT use:**
- "We proved..."
- "This guarantees..."
- "Google's algorithm does..."

In [10]:
# Confirm no private data is in this notebook
print("✓ No client names in this notebook")
print("✓ No real URLs or domains in this notebook")
print("✓ No hardcoded secrets or credentials")
print("✓ Using only anonymized starter CSV")
print("✓ Ready to work on a public-safe analysis")

✓ No client names in this notebook
✓ No real URLs or domains in this notebook
✓ No hardcoded secrets or credentials
✓ Using only anonymized starter CSV
✓ Ready to work on a public-safe analysis


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.